In [8]:
# ============================================================
# PDF -> RAP IA COMPLETO SINCRONIZADO AL BEAT
#
# REQUISITOS:
#
# pip install pymupdf transformers torch torchaudio TTS pydub librosa soundfile
#
# IMPORTANTE:
# Instalar FFmpeg:
# https://ffmpeg.org/download.html
#
# ESTE SCRIPT:
# 1. Extrae texto del PDF
# 2. Usa un LLM para generar MUCHAS barras estilo rap
# 3. Detecta duración del beat
# 4. Genera suficientes líneas para TODA la canción
# 5. Sincroniza frases al BPM
# 6. Mezcla todo automáticamente
#
# ============================================================

import fitz
import os
import math
import librosa

from transformers import pipeline
from TTS.api import TTS
from pydub import AudioSegment

# ============================================================
# CONFIG
# ============================================================

PDF_PATH = "transformers_teoria.pdf"
BEAT_PATH = "example.mp3"

TEMP_FOLDER = "temp_audio"

FINAL_OUTPUT = "rap_final.mp3"

# BPM del beat
BPM = 90

# beats por frase
BEATS_PER_LINE = 2

# ============================================================
# EXTRAER TEXTO PDF
# ============================================================

def extraer_texto(pdf_path):

    texto = ""

    doc = fitz.open(pdf_path)

    for pagina in doc:
        texto += pagina.get_text()

    doc.close()

    return texto


# ============================================================
# DURACION DEL BEAT
# ============================================================

def obtener_duracion_audio(audio_path):

    audio = AudioSegment.from_file(audio_path)

    return len(audio) / 1000


# ============================================================
# GENERAR LETRA LARGA
# ============================================================

def generar_letra_larga(texto, num_lineas):

    generador = pipeline(
        "text-generation",
        model="datificate/gpt2-small-spanish"
    )

    prompt = f"""
Escribe un rap en español.

Tema:
{texto[:800]}

Reglas:
- 1 línea por verso
- estilo urbano
- rimas simples
- exactamente {num_lineas} líneas
"""

    resultado = generador(
        prompt,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.8,
        truncation=True
    )

    texto_generado = resultado[0]["generated_text"]

    lineas = []

    for l in texto_generado.split("\n"):
        l = l.strip()
        if len(l) > 3 and not l.startswith("Escribe"):
            lineas.append(l)

    while len(lineas) < num_lineas:
        lineas.append("flow en la calle sin miedo al mañana")

    return lineas[:num_lineas]


# ============================================================
# GENERAR VOCES
# ============================================================

def generar_voces(lineas):

    if not os.path.exists(TEMP_FOLDER):
        os.makedirs(TEMP_FOLDER)

    tts = TTS(
        model_name="tts_models/es/css10/vits"
    )

    archivos = []

    for i, linea in enumerate(lineas):

        print(f"Generando voz {i+1}/{len(lineas)}")

        path = f"{TEMP_FOLDER}/linea_{i}.wav"

        tts.tts_to_file(
            text=linea,
            file_path=path
        )

        archivos.append(path)

    return archivos


# ============================================================
# CREAR RAP
# ============================================================

def crear_rap(beat_path, archivos_voz):

    beat = AudioSegment.from_file(beat_path)

    # bajar beat
    beat = beat - 7

    resultado = beat

    ms_por_beat = 60000 / BPM

    posicion = 0

    for archivo in archivos_voz:

        voz = AudioSegment.from_file(archivo)

        resultado = resultado.overlay(
            voz,
            position=int(posicion)
        )

        posicion += ms_por_beat * BEATS_PER_LINE

    resultado.export(
        FINAL_OUTPUT,
        format="mp3"
    )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    print("\n==============================")
    print("PDF -> RAP IA")
    print("==============================\n")

    # --------------------------------------------------------
    # EXTRAER TEXTO
    # --------------------------------------------------------

    print("Extrayendo texto PDF...")

    texto = extraer_texto(PDF_PATH)

    # --------------------------------------------------------
    # DURACION BEAT
    # --------------------------------------------------------

    print("Calculando duración beat...")

    duracion = obtener_duracion_audio(BEAT_PATH)

    print(f"Duración beat: {duracion:.2f} segundos")

    # --------------------------------------------------------
    # CALCULAR LINEAS NECESARIAS
    # --------------------------------------------------------

    segundos_por_linea = (60 / BPM) * BEATS_PER_LINE

    num_lineas = math.ceil(
        duracion / segundos_por_linea
    )

    print(f"Lineas necesarias: {num_lineas}")

    # --------------------------------------------------------
    # GENERAR LETRA
    # --------------------------------------------------------

    print("Generando letra IA...")

    lineas = generar_letra_larga(
        texto,
        num_lineas
    )

    print("\n==============================")
    print("LETRA")
    print("==============================\n")

    for l in lineas[:10]:
        print(l)

    print("\n...")

    # --------------------------------------------------------
    # GENERAR VOCES
    # --------------------------------------------------------

    print("\nGenerando voces IA...")

    archivos = generar_voces(lineas)

    # --------------------------------------------------------
    # CREAR RAP
    # --------------------------------------------------------

    print("\nMezclando canción...")

    crear_rap(
        BEAT_PATH,
        archivos
    )

    print("\n==============================")
    print("RAP GENERADO")
    print("==============================")
    print(FINAL_OUTPUT)


PDF -> RAP IA

Extrayendo texto PDF...
Calculando duración beat...
Duración beat: 214.05 segundos
Lineas necesarias: 161
Generando letra IA...


C:\Users\luigu\anaconda3\envs\MusicaP\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



LETRA

- nacional más- en el trabajo de las enfro de los, que deses enfro el mecanismo
- un, oto este, la puede,
- oto es la carne (i have to have access to
- a lot to do)
- línea, en enfro
- oto es, oto el trabajo de las enfro de los,
(borders, pueblos, the city,
where I'm at:
- some places where I
- can have access to),

...

Generando voces IA...
 > tts_models/es/css10/vits is already downloaded.
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | >